<a href="https://colab.research.google.com/github/walidsafaa/OSU/blob/main/Torrent_To_Google_Drive_Downloader_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Torrent To Google Drive Downloader v2

### Mount Google Drive
To stream files we need to mount Google Drive.

In [1]:
# ==============================================================================
# 1. FIX PIP METADATA BUG & INSTALL LIBTORRENT
# ==============================================================================
import os

# Clean up broken metadata files
!rm -rf /usr/lib/python3*/dist-packages/*libtorrent*
!rm -rf /usr/local/lib/python3*/dist-packages/*libtorrent*
!apt-get remove --purge -y python3-libtorrent > /dev/null 2>&1

# Install clean PyPI wheel for Python 3.12
!pip install --no-cache-dir libtorrent


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 20.4 MB/s  0:00:00


### Code to download torrent
Variable **link** stores the link string.

In [4]:

# ==============================================================================
# 2. MOUNT DRIVE & SETUP DIRECTORIES
# ==============================================================================
import libtorrent as lt
from google.colab import drive, files
import shutil
import time
import datetime

drive.mount('/content/drive', force_remount=False)

temp_local_path = '/content/downloads/'
final_drive_path = '/content/drive/My Drive/Torrent/'

os.makedirs(temp_local_path, exist_ok=True)
os.makedirs(final_drive_path, exist_ok=True)

# Upload .torrent file
print("Please select your .torrent file to upload:")
uploaded = files.upload()

torrent_file_path = list(uploaded.keys())[0]
print(f"\nLoaded file: {torrent_file_path}")

# ==============================================================================
# 2. PRIVATE TORRENT CONFIGURATION
# ==============================================================================
import libtorrent as lt
import time
import datetime

settings = {
    'user_agent': 'qBittorrent/4.6.3',
    'enable_dht': False,                     # Private tracker requirement
    'enable_lsd': False,
    'listen_interfaces': '0.0.0.0:6881',
    'active_downloads': 5,
    'download_rate_limit': 0,
    'upload_rate_limit': 0
}

ses = lt.session(settings)

# Parse torrent info
info = lt.torrent_info(torrent_file_path)

atp = lt.add_torrent_params()
atp.ti = info
# Save to LOCAL filesystem first to prevent "No such device" mmap error
atp.save_path = temp_local_path

handle = ses.add_torrent(atp)

begin = time.time()
print("Start Time:", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print(f"Downloading: {handle.status().name}")

state_str = ['queued', 'checking', 'downloading metadata',
             'downloading', 'finished', 'seeding', 'allocating']

# ==============================================================================
# 3. DOWNLOAD LOOP
# ==============================================================================
print("\nConnecting to Private Tracker & Peers...\n")

time.sleep(2)

while handle.status().state != lt.torrent_status.seeding:
    s = handle.status()

    # Filter real system errors
    if s.errc and s.errc.value() != 0:
        print(f"\n[Tracker Error #{s.errc.value()}]: {s.errc.message()}")
        break

    print(
        f"\r{s.progress * 100:.2f}% | "
        f"Down: {s.download_rate / 1000:.1f} kB/s | "
        f"Up: {s.upload_rate / 1000:.1f} kB/s | "
        f"Peers: {s.num_peers} | "
        f"Status: {state_str[s.state]}",
        end=""
    )
    time.sleep(3)

end = time.time()
torrent_name = handle.status().name

print(f"\n\nDownload complete locally! Moving {torrent_name} to Google Drive...")

#
# ==============================================================================
# 5. MOVE COMPLETED FILE TO GOOGLE DRIVE
# ==============================================================================
local_file_location = os.path.join(temp_local_path, torrent_name)
drive_file_location = os.path.join(final_drive_path, torrent_name)

if os.path.exists(local_file_location):
    shutil.move(local_file_location, drive_file_location)
    print(f"Successfully moved to: {drive_file_location}")
else:
    print(f"File moved to Drive: {drive_file_location}")

print(f"Total Elapsed Time: {int((end-begin)//60)} min : {int((end-begin)%60)} sec")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Please select your .torrent file to upload:


Saving [www.arabp2p.net]_-_حياة الامبرطور الجديدة-The Emperor's New Groove.torrent to [www.arabp2p.net]_-_حياة الامبرطور الجديدة-The Emperor's New Groove.torrent

Loaded file: [www.arabp2p.net]_-_حياة الامبرطور الجديدة-The Emperor's New Groove.torrent
Start Time: 2026-07-28 23:21:49
Downloading: [El_Magnifico951]  حياة الامبرطور الجديدة

Connecting to Private Tracker & Peers...

0.00% | Down: 0.0 kB/s | Up: 0.0 kB/s | Peers: 0 | Status: downloading

KeyboardInterrupt: 